# Mini Project: Sentiment Assistant with BERT Fine-Tuning

Welcome to the final day mini project!
In this lab you will fine-tune `bert-base-uncased` on movie reviews, evaluate the model, and connect the dots with real customer-support scenarios.

## Prerequisites & Story Setup

- **Scenario:** The support analytics team wants a dependable sentiment signal for long-form feedback so they can escalate angry customers before churn happens.
- **Environment needs:** Python 3.9+, a GPU-enabled runtime (Colab, Kaggle, or a local GPU machine), ~6 GB free VRAM.
- **Packages:** `torch`, `transformers`, `datasets`, `accelerate`, `evaluate`.

In [5]:
%pip install -q transformers datasets accelerate evaluate
# %pip uninstall torch -y
# %pip install torch --index-url https://download.pytorch.org/whl/cu128

Note: you may need to restart the kernel to use updated packages.


## Imports & Hardware Check

We always begin by confirming versions and hardware. If you see `GPU available: False`, switch the runtime (Runtime → Change runtime type → GPU).

In [1]:
import platform
import torch
from transformers import BertTokenizer, BertForSequenceClassification

print(torch.__version__)
print(torch.cuda.is_available())

print("Python version      :", platform.python_version())
print("PyTorch version     :", torch.__version__)
print("GPU available       :", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

c:\Users\avigd\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.11.0+cu128
True
Python version      : 3.11.5
PyTorch version     : 2.11.0+cu128
GPU available       : True


## Load the IMDB Reviews Dataset

We rely on IMDB because it is balanced (25k positive / 25k negative) and already split.

In [3]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")
print(dataset)

c:\Users\avigd\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\avigd\.cache\huggingface\hub\datasets--stanfordnlp--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\avigd\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_d

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


💬 Hugging Face `datasets` returns a `DatasetDict` with `train` and `test` splits. Each row is a `(text, label)` pair, exactly what our model expects.

Take a quick peek at samples to make sentiment concrete:

In [7]:
for sample in dataset["train"].shuffle(seed=42).select(range(5)):
    print("Label:", "Positive" if sample["label"] else "Negative")
    print(sample["text"][:250], "...\n")

Label: Positive
There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier  ...

Label: Positive
This movie is a great. The plot is very true to the book which is a classic written by Mark Twain. The movie starts of with a scene where Hank sings a song with a bunch of kids called "when you stub your toe on the moon" It reminds me of Sinatra's so ...

Label: Negative
George P. Cosmatos' "Rambo: First Blood Part II" is pure wish-fulfillment. The United States clearly didn't win the war in Vietnam. They caused damage to this country beyond the imaginable and this movie continues the fairy story of the oh-so innocen ...

Label: Positive
In the process of trying to establish the audiences' empathy with Jake Roedel (Tobey Maguire) the filmmakers slander the North and the Jayhawkers. Missouri never withdr

## Tokenizer Setup & Data Pipeline

BERT uses WordPiece tokenization to handle rare or unknown words by breaking them into subword units. It adds `[CLS]` and `[SEP]` tokens, and attention masks tell the model which tokens are real versus padding.

In [8]:
MAX_LENGTH = 256   # trim or pad every review to 256 tokens so batches align
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)

Tokenizer loaded: bert-base-uncased


🧠 we import this tokenizer to reuse the same vocabulary the base model learned in 2018

Next, convert raw text → token IDs, attention masks, and segment IDs (token type IDs).

In [9]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
    )

encoded = dataset.map(tokenize_batch, batched=True)
encoded = encoded.rename_column("label", "labels")
encoded.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids", "labels"],
)

train_ds = encoded["train"]
test_ds  = encoded["test"]

Map: 100%|██████████| 50000/50000 [00:18<00:00, 2743.12 examples/s]


🗒️ `datasets.map(batched=True)` keeps Hugging Face tokenization inside an efficient, cached pipeline, so we don't manually juggle NumPy arrays. `set_format("torch")` hands us tensors ready for training.

## Initialize the Fine-Tuning Model

Now we load `BertForSequenceClassification`, which already bundles the encoder + classification head.

In [10]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
)
model.to(device)
model.config

c:\Users\avigd\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\avigd\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 649.41it/s]
[transformers] BertForSequenc

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.19.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

💡 we import this model to reuse the 110M parameters learned on BooksCorpus + Wikipedia. We only fine-tune for a couple of epochs, which is why learning rates in the 2e-5 range are standard.

## Train and Monitor

On a T4 GPU (the one you find in Google Colab), two epochs take ~15 minutes. Watch both training and validation accuracy. The Hugging Face `Trainer` sets up the AdamW optimizer, the loss, and the accuracy metric for us.

**#To-Do:** complete the `Trainer(...)` call.

In [14]:
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

EPOCHS = 3  # increase to 3 if time allows

training_args = TrainingArguments(
    output_dir="./bert-imdb",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

history = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.229028,0.226203,0.911680
2,0.148476,0.291655,0.921920
3,0.079381,0.344609,0.925280


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.78it/s]


📈 Point out how the validation accuracy plateau indicates when to stop. Encourage screenshots of the learning curves for portfolios.

## Evaluate on the Held-Out Test Set

Even though the `Trainer` already reports validation metrics, we re-run evaluation on the untouched test pipeline to mimic production QA.

**#To-Do:** evaluate on the test set.

In [15]:
eval_metrics = trainer.evaluate(eval_dataset=test_ds)
print(eval_metrics)

Training Loss,Validation Loss,Epoch,Accuracy
0.079381,0.344609,3,0.925280


{'eval_loss': 0.34460872411727905, 'eval_accuracy': 0.92528}


✅ Here you will notice whether accuracy crosses the ~0.90 classroom benchmark.

**#To-Do:** Use this to discuss acceptable error rates for real support teams.

## Build a Reusable Inference Helper

Wrap everything in a function so that you can paste real support transcripts and get an instant score.

**#To-Do:** complete `predict_sentiment`.

In [16]:
def predict_sentiment(text: str):
    inputs = tokenizer(
        text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)

    label_id = int(torch.argmax(probs, dim=-1).item())
    label = "Positive" if label_id == 1 else "Negative"
    return label, float(probs.max())

custom_sentence = "The onboarding emails were confusing, but the agent fixed everything politely."
label, confidence = predict_sentiment(custom_sentence)
print(f"Prediction: {label} (confidence={confidence:.3f})")

Prediction: Positive (confidence=0.944)


For the previous example, you should see `Prediction: Positive (confidence=0.5) slightly more or less`.

🧭 confidence scores are essential when deciding whether to auto-respond or escalate to a human.

## Reflection & Next Steps

1. **Why fine-tuning matters:** You reused a public checkpoint to hit >90% accuracy with minimal data.
2. **Transferable skills:** Everything here also applies to classification tasks in HR, legal, or product analytics.
3. **What you can do with this:** domain adaptation, multilingual checkpoints (DistilBERT multilingual, XLM-R), and monitoring (log data drift, build dashboards).

**#To-Do:** Answer the following reflection questions in markdown cells:

- What lever (data cleaning, hyperparameters, more epochs) most improved results?
- > data cleaning? IMDB datas are already clean and balanced
- > more epochs? it will more overfit the model than improve the accuracy
- > hyperparameters? we are fine-tuning, so parameters like learning_rate is really important (higher than 2e-5 will destroy the trained model)
- Where would you add guardrails before deploying this sentiment signal live?
- > if confidence is not very high (<~70), maybe ask confirmation to a human person
- Which stakeholders benefit the most (support lead, product manager, compliance officer)?
- > for that case, it benefits clearly the most to the support lead